# Particle 2: periodicity with BM4 and 160 steps per cycle

Only particle 2 from the original horizontal study is integrated, using **BM4Implicit only**.
The horizon remains **50 forcing cycles**, with **160 complete integration steps per cycle**:
$h=1/160=0.00625$, 8,000 steps and 8,001 saved states. Every integration step is saved.
The analysis concerns returns and repetition of the trajectory. It does not run other
integrators or a reference-accuracy comparison.

Use the project's `.venv` kernel and run all cells. The first complete calculation can
take several minutes. Its separate archive is reused on subsequent runs when the
configuration and input identity match. Set `reuse_saved_result=False` to recompute.
The original notebooks and archives are preserved.

## State, forcing phase and periodicity

The initial state is the original **particle 2**, array index 1:
$(x_0,y_0)=(x_{\min}+0.525L,y_{\min}+0.5L)$, where $L$ is the periodic cell width.
The 20 original guiding centres are independent, so extracting this one particle
preserves its equations. The state is $(x,y)$; velocity follows from the first-order
GC equations $\dot x=-\partial_y\bar\Phi_\rho$, $\dot y=\partial_x\bar\Phi_\rho$.
The gyro-radius is $\rho=0.3$. No new interaction or initial velocity is specified.

Use the same measured field, mean plus dominant mode, cubic interpolation, magnetic
field 1.5 and characteristic length 0.06. Coordinates are normalized by
$\hat x=2\pi(R-R_0)/0.06$, and time by the dominant-mode period. Its normalized
frequency is one. **Integer times therefore have the same forcing phase.**
The BM4 coupling frequency $\pi/8$ is a numerical parameter.

We measure two different quantities using minimum-image spatial distance:

- $d_n=\|z(n)-z(0)\|_{\rm per}$: return to the initial state after $n$ cycles.
- $E_k(t)=\|z(t+k)-z(t)\|_{\rm per}$: repetition over the overlapping trajectory
  for a candidate integer period $k$.

A small $d_k$ alone does not establish periodicity. Small $E_k(t)$ throughout an
observed overlap supports approximate repetition at the selected tolerance. Longer
lags have shorter overlaps, which are reported explicitly. The test uses the
computed BM4 orbit and does not prove an exact physical periodic orbit.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import csv
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from diagnostics import load_parallel_bm4_recurrence_npz, write_parallel_bm4_recurrence_npz
from diagnostics.paths import find_project_root
from initial_conditions import GCInitialConfiguration
from potential import load_gc2d_h5_potential
from studies.bm4_parallel_recurrence import ParallelBM4RecurrenceConfig, run_parallel_bm4_recurrence
from studies.bm4_periodicity import periodicity_records
from visualization import display_records_table
from visualization.bm4_periodicity import plot_bm4_periodicity

In [2]:
particle_number = 2
cycle_count = 50
steps_per_cycle = 160
saved_samples_per_cycle = 160
rho = 0.3
coupling_frequency = float(np.pi / 8)
newton_atol = 1e-12
newton_rtol = 1e-11
newton_max_iterations = 40
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

potential_specification = dict(
    source_path='data/potential/V1/PHI_2.h5', magnetic_field=1.5,
    characteristic_length=0.06, mode_selection=[0, 1], interpolation_order=3,
)
recurrence_tolerance_fraction = 0.01  # Distance threshold as a fraction of L.
threshold_fractions = (0.01, 0.001, 0.0001)
candidate_cycle = None  # None selects the closest nonzero integer-time return.
reuse_saved_result = True

config = ParallelBM4RecurrenceConfig(
    particle_count=1, t_span=(0., float(cycle_count)),
    steps_per_cycle=steps_per_cycle, saved_samples_per_cycle=saved_samples_per_cycle,
    rho=rho, coupling_frequency=coupling_frequency, absolute_tolerance=newton_atol,
    relative_tolerance=newton_rtol, max_iterations=newton_max_iterations,
    jacobian_relative_step=jacobian_relative_step, worker_count=1, progress=True,
)
assert steps_per_cycle == saved_samples_per_cycle == 160
assert config.integration_step == 0.00625
assert config.step_count == cycle_count * 160
assert 0 < recurrence_tolerance_fraction < 0.5
print(f'{config.step_count} BM4 steps; {config.output_sample_count} saved states.')

8000 BM4 steps; 8001 saved states.


In [3]:
project_root = find_project_root(Path.cwd())
notebook_directory = project_root / 'notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences'
source_path = notebook_directory / 'results.npz'
result_path = notebook_directory / 'particle_2_bm4_160_results.npz'
output_directory = notebook_directory / 'particle_2_bm4_160_outputs'
source = load_parallel_bm4_recurrence_npz(source_path)
assert source.metadata['experiment']['potential'] == potential_specification
assert source.result.config.particle_count == 20 and particle_number == 2
initial_state = source.result.initial_positions[particle_number - 1].copy()
data_path = project_root / potential_specification['source_path']
potential = load_gc2d_h5_potential(
    data_path, B=potential_specification['magnetic_field'],
    characteristic_length=potential_specification['characteristic_length'],
    indx=tuple(potential_specification['mode_selection']),
    interpolation_order=potential_specification['interpolation_order'],
)
period = float(potential.grid.period)
np.testing.assert_allclose(initial_state, [potential.grid.xmin + .525 * period,
                                         potential.grid.ymin + .5 * period], rtol=0, atol=1e-12)
np.testing.assert_allclose(potential.frequencies, [1.], rtol=0, atol=1e-14)
initial_configuration = GCInitialConfiguration.from_components(
    x=initial_state[:1], y=initial_state[1:],
)
# Input and implementation identities guard against accidental reuse after changes.
code_paths = sorted(set(
    list((project_root / 'src/simulation').rglob('*.py'))
    + list((project_root / 'src/potential').glob('*.py'))
    + list((project_root / 'src/dynamics').glob('*.py'))
    + [project_root / 'src/studies/bm4_parallel_recurrence.py']
))
experiment = dict(
    study='Particle 2 BM4 periodicity, 160 steps per cycle', particle_number=particle_number,
    initial_state=initial_state.tolist(), potential=potential_specification,
    source_sha256=hashlib.sha256(source_path.read_bytes()).hexdigest(),
    potential_sha256=hashlib.sha256(data_path.read_bytes()).hexdigest(),
    implementation_sha256={str(path.relative_to(project_root)): hashlib.sha256(path.read_bytes()).hexdigest()
                           for path in code_paths},
)
print(f'Particle 2 initial state: {initial_state}; cell width L={period:.12g}')

Particle 2 initial state: [9.89601686 9.42477796]; cell width L=18.8495559215


## BM4 calculation

The integration starts at time zero from the original initial position. It uses
`float64` arithmetic and the same Newton settings as the source study, with an
analytic Jacobian. The worker count is one. The progress log is printed while the
single orbit runs; no multi-particle campaign is executed.

In [4]:
if reuse_saved_result and result_path.exists():
    saved = load_parallel_bm4_recurrence_npz(result_path)
    if saved.result.config != config or saved.metadata['experiment'] != experiment:
        raise ValueError('Saved calculation settings or inputs changed. Set reuse_saved_result=False to recompute.')
    result = saved.result
    print(f'Loaded {result_path.name}')
else:
    result = run_parallel_bm4_recurrence(potential, initial_configuration, config=config)
    write_parallel_bm4_recurrence_npz(result, result_path, metadata=experiment, overwrite=True)
    print(f'Saved {result_path.name}')

assert result.positions.shape == (1, 2, cycle_count * 160 + 1)
np.testing.assert_array_equal(result.initial_positions[0], initial_state)
np.testing.assert_allclose(result.times, np.arange(cycle_count * 160 + 1) / 160, rtol=0, atol=1e-12)
assert result.maximum_residual_to_tolerance[0] <= 1.0
returns, lags = periodicity_records(result, period=period)
best_return = min(returns, key=lambda row: row['distance'])
if candidate_cycle is None:
    candidate_cycle = best_return['cycle']
if int(candidate_cycle) != candidate_cycle or not 1 <= candidate_cycle <= cycle_count:
    raise ValueError('candidate_cycle must be an integer between 1 and cycle_count.')
candidate_cycle = int(candidate_cycle)

Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 0.0 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 31.3 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 61.3 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 91.3 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 121.3 s; ETA after first completion.


...

Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 271.3 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 301.3 s; ETA after first completion.


Parallel BM4 recurrence: 0/1 trajectories; workers 1; elapsed 331.3 s; ETA after first completion.


Parallel BM4 recurrence: 1/1 trajectories; workers 1; elapsed 346.3 s; ETA 0.0 s.


Saved particle_2_bm4_160_results.npz


## Returns at the initial forcing phase

Cycle zero is excluded. The table lists the ten closest integer-time returns.
`Delta x` and `Delta y` are periodic displacements from the initial position,
not estimates of numerical integration error.

In [5]:
display_records_table([SimpleNamespace(**row) for row in sorted(returns, key=lambda row: row['distance'])[:10]],
    columns=(('cycle', 'Cycle', 'd'), ('distance', 'Distance', '.10e'),
             ('fraction', 'Distance / L', '.6e'), ('dx', 'Delta x', '.8e'), ('dy', 'Delta y', '.8e')))
figure = plot_bm4_periodicity(
    result, period=period, particle_number=particle_number,
    threshold_fraction=recurrence_tolerance_fraction, candidate_cycle=candidate_cycle,
)
plt.show()

Cycle,Distance,Distance / L,Delta x,Delta y
36,8.5757924372e-03,4.549599e-04,-4.62009431e-03,-7.22488370e-03
29,4.2940431053e-02,2.278061e-03,2.77104655e-02,3.28026024e-02
7,5.5972866509e-02,2.969453e-03,-3.18350417e-02,-4.60379398e-02
43,6.8666942737e-02,3.642894e-03,-3.79596600e-02,-5.72207413e-02
22,9.9526976344e-02,5.280070e-03,6.49779768e-02,7.53888689e-02
14,1.3034946903e-01,6.915254e-03,-6.90204037e-02,-1.10576525e-01
50,1.4901442707e-01,7.905461e-03,-7.82978195e-02,-1.26786241e-01
15,1.5795608464e-01,8.379831e-03,8.69198097e-02,1.31890376e-01
44,1.9392844855e-01,1.028822e-02,8.57222125e-02,1.73953860e-01
8,2.0042309217e-01,1.063278e-02,8.36923984e-02,1.82112598e-01


/tmp/ipykernel_55593/2734410115.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Does the trajectory repeat?

For each integer lag $k$, compare **all saved BM4 states** at $t$ and $t+k$ in the
available overlap. The table ranks lags by RMS displacement; the maximum gives the
largest observed failure to repeat. Inspect the overlap duration when interpreting
this ranking. A lag equal to the complete horizon only compares two endpoints and
is excluded from this repetition table.

In [6]:
if lags:
    display_records_table([SimpleNamespace(**row) for row in sorted(lags, key=lambda row: row['rms'])[:10]],
        columns=(('cycle', 'Candidate period', 'd'), ('overlap_cycles', 'Overlap [cycles]', 'd'),
                 ('pairs', 'State pairs', 'd'), ('rms', 'RMS displacement', '.8e'),
                 ('maximum', 'Maximum displacement', '.8e'), ('maximum_fraction', 'Maximum / L', '.6e')))
else:
    print('Increase cycle_count to obtain a positive-duration overlap.')

for fraction in threshold_fractions:
    close_cycles = [row['cycle'] for row in returns if row['fraction'] <= fraction]
    repeating_lags = [row['cycle'] for row in lags if row['maximum_fraction'] <= fraction]
    print(f'Threshold {fraction:g} L: returns to the initial point at cycles {close_cycles}')
    print(f'  Integer lags within the threshold throughout the saved overlap: {repeating_lags}')

selected = next((row for row in lags if row['cycle'] == candidate_cycle), None)
summary = [f"Closest same-phase return: cycle {best_return['cycle']}, distance {best_return['distance']:.8e} "
           f"({best_return['fraction']:.8e} L)."]
if selected is not None:
    summary.append(f"Candidate period {candidate_cycle}: overlap {selected['overlap_cycles']} cycles, "
                   f"maximum displacement {selected['maximum_fraction']:.8e} L.")
    summary.append('This candidate repeats within the chosen threshold over all saved overlapping states.'
                   if selected['maximum_fraction'] <= recurrence_tolerance_fraction else
                   'This candidate does not repeat within the chosen threshold throughout the observed overlap.')
else:
    summary.append('Extend cycle_count beyond the candidate period to test repetition over an interval.')
summary.append('These are finite-horizon BM4 periodicity diagnostics, not a proof of exact periodicity.')
display(Markdown('\n\n'.join(summary)))

Candidate period,Overlap [cycles],State pairs,RMS displacement,Maximum displacement,Maximum / L
36,14,2241,1.11750125e-02,1.89030787e-02,1.002839e-03
29,21,3361,4.70579429e-02,9.41233913e-02,4.993401e-03
7,43,6881,5.82168496e-02,1.19670795e-01,6.348733e-03
43,7,1121,6.78456479e-02,9.54955582e-02,5.066197e-03
22,28,4481,1.04477441e-01,2.13613769e-01,1.133256e-02
14,36,5761,1.14797573e-01,2.30278122e-01,1.221663e-02
15,35,5601,1.58951722e-01,2.90094979e-01,1.539002e-02
21,29,4641,1.68879624e-01,3.04087956e-01,1.613237e-02
44,6,961,2.00888327e-01,3.38820217e-01,1.797497e-02
8,42,6721,2.11321480e-01,3.63392532e-01,1.927857e-02


Threshold 0.01 L: returns to the initial point at cycles [7, 14, 15, 22, 29, 36, 43, 50]
  Integer lags within the threshold throughout the saved overlap: [7, 29, 36, 43]
Threshold 0.001 L: returns to the initial point at cycles [36]
  Integer lags within the threshold throughout the saved overlap: []
Threshold 0.0001 L: returns to the initial point at cycles []
  Integer lags within the threshold throughout the saved overlap: []


Closest same-phase return: cycle 36, distance 8.57579244e-03 (4.54959919e-04 L).

Candidate period 36: overlap 14 cycles, maximum displacement 1.00283947e-03 L.

This candidate repeats within the chosen threshold over all saved overlapping states.

These are finite-horizon BM4 periodicity diagnostics, not a proof of exact periodicity.

## Save the periodicity results

The NPZ contains the one-particle BM4 calculation. The CSV files retain every
integer return and every tested integer lag. Only this notebook's own output
files are replaced when these cells are rerun.

In [7]:
output_directory.mkdir(parents=True, exist_ok=True)
for name, rows in (('integer_returns', returns), ('integer_period_tests', lags)):
    if rows:
        with (output_directory / f'{name}.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
figure.savefig(output_directory / 'periodicity.png', dpi=180, bbox_inches='tight')
figure.savefig(output_directory / 'periodicity.svg', bbox_inches='tight')
(output_directory / 'summary.txt').write_text('\n'.join(summary) + '\n', encoding='utf-8')
print(f'BM4 archive: {result_path}')
print(f'Periodicity tables and figure: {output_directory}')

BM4 archive: /home/juan/Proyectos/GC2D_intranet/notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences/particle_2_bm4_160_results.npz
Periodicity tables and figure: /home/juan/Proyectos/GC2D_intranet/notebooks/developements/recurrences/study_20_horizontal_bm4_recurrences/particle_2_bm4_160_outputs
